In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

df.tail(10)

,0Y1M,0Y3M,0Y6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
Date,,,,,,,,,,,
2026-05-01,3.71,3.68,3.71,3.73,3.88,3.91,4.02,4.20,4.39,4.96,4.97
2026-05-04,3.71,3.70,3.76,3.78,3.95,3.98,4.08,4.26,4.45,5.01,5.02
2026-05-05,3.70,3.69,3.75,3.77,3.93,3.97,4.08,4.25,4.43,4.98,4.98
2026-05-06,3.70,3.69,3.74,3.73,3.87,3.89,3.99,4.17,4.36,4.92,4.94
2026-05-07,3.72,3.69,3.74,3.76,3.92,3.94,4.04,4.22,4.41,4.96,4.97
2026-05-08,3.71,3.69,3.74,3.75,3.90,3.92,4.02,4.19,4.38,4.93,4.95
2026-05-11,3.71,3.70,3.77,3.79,3.95,3.96,4.07,4.24,4.42,4.97,4.98
2026-05-12,3.71,3.70,3.77,3.80,4.00,4.01,4.12,4.29,4.46,5.02,5.03
2026-05-13,3.71,3.69,3.77,3.79,3.98,4.00,4.12,4.28,4.46,5.03,5.03


In [3]:
"""
L(t) = 1
S(t) = 1-e^(-xt)/xt
C(t) = S(t) - e^(-xt) 

x = 0.0609
"""

maturities = [1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]
x = 0.0609
A = np.array([
    [
    1,
    (1 - np.exp(-x * t)) / (x * t),
    ((1 - np.exp(-x * t)) / (x * t)) - np.exp(-x * t),
    ]
    for t in maturities
])


rows = []

for row in df.itertuples():
    # get actual yields for each date
    y = [row._1, row._2, row._3, row._4, row._5, row._6, row._7, row._8, row._9, row._10, row._11]

    betas, *_ = np.linalg.lstsq(A, y, rcond=None)
    b1, b2, b3 = betas
    rows.append({'Date': row.Index, 'Beta 1': b1, 'Beta 2': b2, 'Beta 3': b3})

betas = pd.DataFrame(rows).set_index("Date")
# betas.shift(-5)
betas.tail()



,Beta 1,Beta 2,Beta 3
Date,,,
2026-05-08,5.270308,-1.599870,1.604442
2026-05-11,5.058645,-1.366787,2.039306
2026-05-12,4.857632,-1.157728,2.533876
2026-05-13,4.870180,-1.178757,2.539232
2026-05-14,4.700093,-1.001930,2.771229


In [4]:
# Evaluate exactly 20 expanding, one-day-ahead rolling forecasts.
rolling_windows = 20
forecast_indexes = range(len(betas) - rolling_windows, len(betas))

def rolling_metrics(errors, predicted_changes, actual_changes):
    rmse_basis_points = 100 * np.sqrt(np.mean(np.asarray(errors) ** 2))
    directional_accuracy = 100 * np.mean(
        np.sign(predicted_changes) == np.sign(actual_changes)
    )
    return rmse_basis_points, directional_accuracy

In [5]:
beta_metrics = []

for beta_index in range(3):
    errors = []
    predicted_changes = []
    actual_changes = []

    for forecast_index in forecast_indexes:
        history = betas.iloc[:forecast_index, beta_index].to_numpy()
        x_train = np.column_stack((np.ones(len(history) - 1), history[:-1]))
        y_train = history[1:]
        coeffs, *_ = np.linalg.lstsq(x_train, y_train, rcond=None)

        current = history[-1]
        predicted = np.array([1, current]) @ coeffs
        actual = betas.iloc[forecast_index, beta_index]

        errors.append(predicted - actual)
        predicted_changes.append(predicted - current)
        actual_changes.append(actual - current)

    beta_metrics.append(
        rolling_metrics(errors, predicted_changes, actual_changes)
    )

In [6]:
print(f"Rolling windows: {rolling_windows}")
for beta, (beta_rmse, directional_accuracy) in enumerate(beta_metrics, start=1):
    print(f"  Beta: {beta}")
    print(
        f"  Forecast 1 day, RMSE: {beta_rmse} basis points, "
        f"Directional Accuracy: {directional_accuracy}%"
    )

Rolling windows: 20
  Beta: 1
  Forecast 1 day, RMSE: 25.911209984958028 basis points, Directional Accuracy: 60.0%
  Beta: 2
  Forecast 1 day, RMSE: 27.022152557011665 basis points, Directional Accuracy: 65.0%
  Beta: 3
  Forecast 1 day, RMSE: 49.857715423300434 basis points, Directional Accuracy: 65.0%
